# MecâniQA — OAT 3 | Encontros 30/09 e 07/10: Tunando o Meu Modelo

**Time:** Boa Vista — Alex Oliveira Santos, Alice Gomes Aragão, Ana Clara Ribeiro da Silva, Lorran Edley Matos Ribeiro, Rodrigo dos Anjos Lamartine

**Este notebook é cumulativo:** as seções 0 a 5 são o encontro de **30/09** (modelos com hiperparâmetros padrão contra os baselines) e a seção 6 em diante é o encontro de **07/10** (Grid Search integrado ao `TimeSeriesSplit`).

## Decisões da equipe (brainstorm)

**1. Escolha do algoritmo:** Random Forest e Gradient Boosting, com Árvore de Decisão e Regressão Linear como referência. Modelos de árvores capturam padrões não lineares, como o pico de demanda no fim de semana e a tendência de crescimento da oficina, sem precisar transformar os dados. O Random Forest treina várias árvores e tira a média das previsões, o que reduz a variância. O Gradient Boosting treina árvores em sequência, e cada uma corrige o erro da anterior.

**2. Linha do tempo:** a divisão é cronológica, sem embaralhar os dias: os primeiros 80% para treino e os 20% finais para teste. O `fit()` usa somente o treino, e o teste só entra no `predict()` para medir o erro. As features de lag e rolling usam `shift`, então cada linha só carrega informação do passado.

## 0. Preparação: importações, dados limpos e features (OAT 1 e 2)

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from IPython.display import Markdown, display
import time

SEED = 42
ALVOS = ["Trocas_Oleo", "Manutencao_Motor"]
NOMES = {"Trocas_Oleo": "Trocas de Óleo", "Manutencao_Motor": "Manutenção de Motor"}
pd.options.display.float_format = "{:.2f}".format

# --- Dataset limpo (pipeline da OAT 1) ---
df = pd.read_excel("mecaniqa_dataset.xlsx")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")

for alvo in ALVOS:
    df.loc[df[alvo] < 0, alvo] = np.nan                  # negativos são inválidos
    df[alvo] = df[alvo].interpolate(method="time")       # ausentes e inválidos
    q1, q3 = df[alvo].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[alvo] = df[alvo].clip(lower=max(0, q1 - 1.5 * iqr), upper=q3 + 1.5 * iqr)

# --- Features temporais (OAT 2): lags 1, 7, 30 e rolling mean de 7 dias (só passado) ---
df_feat = df.copy()
for alvo in ALVOS:
    for lag in [1, 7, 30]:
        df_feat[f"{alvo}_lag_{lag}"] = df_feat[alvo].shift(lag)
    df_feat[f"{alvo}_rolling_mean_7"] = df_feat[alvo].rolling(7, min_periods=7).mean().shift(1)

df_feat = df_feat.dropna()
FEATURES = [c for c in df_feat.columns if c not in ALVOS]
print(f"Dataset de features: {df_feat.shape[0]} dias, {len(FEATURES)} features")

Dataset de features: 701 dias, 8 features


## 1. Divisão temporal (treino no passado, teste no futuro)

In [2]:
n_treino = int(len(df_feat) * 0.80)
treino, teste = df_feat.iloc[:n_treino], df_feat.iloc[n_treino:]
X_treino, X_teste = treino[FEATURES], teste[FEATURES]

assert treino.index.max() < teste.index.min(), "O treino precisa terminar antes do teste!"
print(f"Treino: {treino.index.min().date()} a {treino.index.max().date()}  ({len(treino)} dias)")
print(f"Teste : {teste.index.min().date()} a {teste.index.max().date()}  ({len(teste)} dias)")

Treino: 2024-01-31 a 2025-08-12  (560 dias)
Teste : 2025-08-13 a 2025-12-31  (141 dias)


## 2. Baselines antigos (referência a ser batida) e função de erro

In [3]:
def avaliar_metricas(y_real, y_previsto):
    y_real, y_prev = np.asarray(y_real, dtype=float), np.asarray(y_previsto, dtype=float)
    mascara = y_real != 0   # MAPE não é definido quando o valor real é zero
    return {
        "RMSE": np.sqrt(mean_squared_error(y_real, y_prev)),
        "MAPE": mean_absolute_percentage_error(y_real[mascara], y_prev[mascara]) * 100,
        "MAE": mean_absolute_error(y_real, y_prev),
    }


# Baselines da OAT 2 (shift(1) evita olhar o dia atual)
def previsao_naive(serie):
    return serie.shift(1)

def previsao_media_movel(serie, janela=7):
    return serie.rolling(janela, min_periods=janela).mean().shift(1)


# Baseline antigo de cada variável = o de menor MAE no TREINO (sem olhar o teste)
baseline_prev, baseline_nome = {}, {}
for alvo in ALVOS:
    candidatos = {"Naive": previsao_naive(df[alvo]), "Média Móvel 7d": previsao_media_movel(df[alvo])}
    mae_treino = {n: avaliar_metricas(df[alvo].loc[treino.index], p.loc[treino.index])["MAE"] for n, p in candidatos.items()}
    baseline_nome[alvo] = min(mae_treino, key=mae_treino.get)
    baseline_prev[alvo] = candidatos[baseline_nome[alvo]].loc[teste.index]
    print(f"{NOMES[alvo]}: baseline antigo = {baseline_nome[alvo]}")

Trocas de Óleo: baseline antigo = Naive
Manutenção de Motor: baseline antigo = Média Móvel 7d


## 3. Passo 1 — Instanciar modelos clássicos de regressão (hiperparâmetros default)

In [4]:
def criar_modelos():
    return {
        "Regressão Linear": LinearRegression(),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
        "Random Forest": RandomForestRegressor(random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
    }

modelos = {alvo: criar_modelos() for alvo in ALVOS}
print("Modelos instanciados:", list(modelos["Trocas_Oleo"].keys()))

Modelos instanciados: ['Regressão Linear', 'Árvore de Decisão', 'Random Forest', 'Gradient Boosting']


## 4. Passo 2 e 3 — Pipeline: `fit()` só no treino e `predict()` no teste

In [5]:
previsoes = {alvo: {} for alvo in ALVOS}

for alvo in ALVOS:
    for nome, modelo in modelos[alvo].items():
        modelo.fit(X_treino, treino[alvo])                       # fit EXCLUSIVAMENTE no treino
        previsoes[alvo][nome] = modelo.predict(X_teste)          # predict no teste, com hiperparâmetros default

print("Treino e previsões concluídos para todos os modelos.")

Treino e previsões concluídos para todos os modelos.


## 5. Critério de aceite — erro inicial (RMSE e MAPE) dos modelos "crus" contra o baseline antigo

In [6]:
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    print(f"\n=== {NOMES[alvo]} ===")
    print(f"Baseline antigo ({baseline_nome[alvo]}) - RMSE: {b['RMSE']:.2f}, MAPE: {b['MAPE']:.2f}%")
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "BATE o baseline" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não bate o baseline"
        print(f"{nome} - RMSE: {m['RMSE']:.2f}, MAPE: {m['MAPE']:.2f}%  ->  {bate}")


=== Trocas de Óleo ===
Baseline antigo (Naive) - RMSE: 8.92, MAPE: 26.80%
Regressão Linear - RMSE: 4.16, MAPE: 14.02%  ->  BATE o baseline
Árvore de Decisão - RMSE: 5.85, MAPE: 20.38%  ->  BATE o baseline
Random Forest - RMSE: 3.81, MAPE: 13.34%  ->  BATE o baseline
Gradient Boosting - RMSE: 4.00, MAPE: 13.70%  ->  BATE o baseline

=== Manutenção de Motor ===
Baseline antigo (Média Móvel 7d) - RMSE: 3.82, MAPE: 41.05%
Regressão Linear - RMSE: 2.42, MAPE: 26.96%  ->  BATE o baseline
Árvore de Decisão - RMSE: 3.90, MAPE: 38.37%  ->  não bate o baseline
Random Forest - RMSE: 2.48, MAPE: 28.40%  ->  BATE o baseline
Gradient Boosting - RMSE: 2.49, MAPE: 28.45%  ->  BATE o baseline


### Tabela-resumo

In [7]:
linhas = []
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    linhas.append({"Variável": NOMES[alvo], "Modelo": f"Baseline antigo ({baseline_nome[alvo]})", "RMSE": b["RMSE"], "MAPE (%)": b["MAPE"], "Bate o baseline?": "referência"})
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "SIM" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não"
        linhas.append({"Variável": NOMES[alvo], "Modelo": nome, "RMSE": m["RMSE"], "MAPE (%)": m["MAPE"], "Bate o baseline?": bate})

pd.DataFrame(linhas).set_index(["Variável", "Modelo"])

RMSE  ...  Bate o baseline?
Variável            Modelo                                  ...                  
Trocas de Óleo      Baseline antigo (Naive)           8.92  ...        referência
                    Regressão Linear                  4.16  ...               SIM
                    Árvore de Decisão                 5.85  ...               SIM
                    Random Forest                     3.81  ...               SIM
                    Gradient Boosting                 4.00  ...               SIM
Manutenção de Motor Baseline antigo (Média Móvel 7d)  3.82  ...        referência
                    Regressão Linear                  2.42  ...               SIM
                    Árvore de Decisão                 3.90  ...               não
                    Random Forest                     2.48  ...               SIM
                    Gradient Boosting                 2.49  ...               SIM

[10 rows x 3 columns]

## 6. Encontro 07/10 — Grid Search integrado ao TimeSeriesSplit

**Objetivo prático:** codificar a otimização de parâmetros e registrar os ganhos em relação ao modelo treinado na semana passada (Random Forest com hiperparâmetros default).

### Decisões da equipe (brainstorm)

**1. Espaço de busca:** vamos sintonizar o Random Forest, que teve o menor erro de validação entre os modelos de árvores (comprovado na seção 6.1). Os hiperparâmetros e limites são `n_estimators` em {50, 100, 200}, porque mais árvores estabilizam a média, com retorno decrescente; `max_depth` em {3, 5, 10, None}, que controla a complexidade (raso demais não aprende o padrão, fundo demais decora o treino); e `min_samples_leaf` em {1, 2, 4, 8}, porque folhas maiores suavizam o ruído. São 48 combinações e, com 5 folds, 240 treinos, um custo razoável para 560 dias de treino.

**2. Proteção temporal:** o Grid Search divide os dados várias vezes. Com um K-Fold comum, os dias seriam embaralhados e o modelo treinaria com o futuro para prever o passado, o que dá um erro otimista e escolhe hiperparâmetros errados. Com o `TimeSeriesSplit`, cada fold treina no passado e valida no futuro, então o `best_params_` reflete como o modelo se comportaria na prática. A busca roda só no treino, e o teste fica intocado para a avaliação final.

### 6.1 Qual é o nosso melhor modelo? (escolha feita só no treino, sem olhar o teste)

In [8]:
tscv = TimeSeriesSplit(n_splits=5)

candidatos = {
    "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
    "Random Forest": RandomForestRegressor(random_state=SEED),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
}

cv_inicial = {alvo: {} for alvo in ALVOS}
for alvo in ALVOS:
    for nome, modelo in candidatos.items():
        scores = cross_val_score(modelo, X_treino, treino[alvo], cv=tscv,
                                 scoring="neg_root_mean_squared_error", n_jobs=-1)
        cv_inicial[alvo][nome] = -scores.mean()

tabela_cv = pd.DataFrame(cv_inicial).rename(columns=NOMES)
tabela_cv.index.name = "RMSE na validação cruzada (treino)"
for alvo in ALVOS:
    print(f"{NOMES[alvo]:<20} menor RMSE na validação: {min(cv_inicial[alvo], key=cv_inicial[alvo].get)}")
tabela_cv

Trocas de Óleo       menor RMSE na validação: Random Forest
Manutenção de Motor  menor RMSE na validação: Random Forest


,Trocas de Óleo,Manutenção de Motor
RMSE na validação cruzada (treino),,
Árvore de Decisão,9.67,3.83
Random Forest,5.86,3.00
Gradient Boosting,7.46,3.56


### 6.2 Passos 1 e 2 — Dicionário de hiperparâmetros e `GridSearchCV` com `TimeSeriesSplit`

In [9]:
# Passo 1: dicionário de hiperparâmetros (o "grid")
grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 2, 4, 8],
}
n_combinacoes = int(np.prod([len(v) for v in grid.values()]))
print(f"Grid: {n_combinacoes} combinações x 5 folds = {n_combinacoes * 5} treinos por variável\n")

# Passo 2 e 3: GridSearchCV com TimeSeriesSplit dentro da validação cruzada
grid_busca = {}
for alvo in ALVOS:
    busca = GridSearchCV(
        estimator=RandomForestRegressor(random_state=SEED),
        param_grid=grid,
        cv=TimeSeriesSplit(n_splits=5),            # integração obrigatória com a linha do tempo
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    inicio = time.perf_counter()
    busca.fit(X_treino, treino[alvo])               # busca feita SOMENTE no treino
    duracao = time.perf_counter() - inicio
    grid_busca[alvo] = busca

    print(f"=== {NOMES[alvo]} ===")
    print("best_params_:", busca.best_params_)
    print(f"Melhor RMSE na validação cruzada: {-busca.best_score_:.3f}  | tempo: {duracao:.1f} s\n")

Grid: 48 combinações x 5 folds = 240 treinos por variável

=== Trocas de Óleo ===
best_params_: {'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 50}
Melhor RMSE na validação cruzada: 5.200  | tempo: 34.6 s

=== Manutenção de Motor ===
best_params_: {'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 100}
Melhor RMSE na validação cruzada: 2.757  | tempo: 33.8 s



### 6.3 Critério de aceite — redução do erro em relação ao modelo da semana passada

In [10]:
ganho = {}
for alvo in ALVOS:
    prev_inicial = previsoes[alvo]["Random Forest"]                       # modelo da semana passada (default)
    prev_tunado = grid_busca[alvo].best_estimator_.predict(X_teste)       # modelo tunado
    m_ini = avaliar_metricas(teste[alvo], prev_inicial)
    m_tun = avaliar_metricas(teste[alvo], prev_tunado)
    cv_ini, cv_tun = cv_inicial[alvo]["Random Forest"], -grid_busca[alvo].best_score_

    tabela = pd.DataFrame(
        {
            "RMSE validação cruzada (treino)": (cv_ini, cv_tun),
            "RMSE teste": (m_ini["RMSE"], m_tun["RMSE"]),
            "MAPE teste (%)": (m_ini["MAPE"], m_tun["MAPE"]),
            "MAE teste": (m_ini["MAE"], m_tun["MAE"]),
        },
        index=["RF inicial (default)", "RF tunado (Grid Search)"],
    ).T
    tabela["Redução do erro (%)"] = (tabela.iloc[:, 0] - tabela.iloc[:, 1]) / tabela.iloc[:, 0] * 100
    ganho[alvo] = tabela

    print(f"\n=== {NOMES[alvo]} ===")
    print("best_params_:", grid_busca[alvo].best_params_)
    print(tabela.to_string())


=== Trocas de Óleo ===
best_params_: {'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 50}
                                 RF inicial (default)  RF tunado (Grid Search)  Redução do erro (%)
RMSE validação cruzada (treino)                  5.86                     5.20                11.30
RMSE teste                                       3.81                     3.82                -0.46
MAPE teste (%)                                  13.34                    13.86                -3.91
MAE teste                                        3.04                     3.13                -2.96

=== Manutenção de Motor ===
best_params_: {'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 100}
                                 RF inicial (default)  RF tunado (Grid Search)  Redução do erro (%)
RMSE validação cruzada (treino)                  3.00                     2.76                 7.99
RMSE teste                                       2.48                     2.39                

### Conclusão dos dois encontros (texto gerado a partir dos resultados desta execução)

In [11]:
def frase_variacao(p):
    return f"redução de {p:.1f}%" if p >= 0 else f"aumento de {abs(p):.1f}%"


partes = ["### Resumo dos encontros 30/09 e 07/10\n"]
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    g = ganho[alvo]
    ini_rmse, tun_rmse = g.loc["RMSE teste"].iloc[0], g.loc["RMSE teste"].iloc[1]
    partes.append(
        f"**{NOMES[alvo]}**\n"
        f"- 30/09: o Random Forest sem ajustes já reduziu o RMSE do baseline de **{b['RMSE']:.2f}** para **{ini_rmse:.2f}**.\n"
        f"- 07/10: o Grid Search escolheu `{grid_busca[alvo].best_params_}`. No erro de validação houve "
        f"**{frase_variacao(g.loc['RMSE validação cruzada (treino)', 'Redução do erro (%)'])}**; no teste, o RMSE foi de {ini_rmse:.2f} para "
        f"**{tun_rmse:.2f}** ({frase_variacao(g.loc['RMSE teste', 'Redução do erro (%)'])}) e o MAPE teve "
        f"{frase_variacao(g.loc['MAPE teste (%)', 'Redução do erro (%)'])}.\n"
    )

ganhos_teste = [ganho[a].loc["RMSE teste", "Redução do erro (%)"] for a in ALVOS]
if all(abs(x) < 5 for x in ganhos_teste):
    partes.append(
        "**Leitura honesta:** o tuning melhora o erro de validação (que é o que a busca otimiza), mas no teste o ganho adicional é pequeno, "
        "sinal de que o Random Forest com hiperparâmetros default já estava perto do teto para estes atributos. "
        "O maior salto de desempenho veio de trocar o baseline por um modelo de Machine Learning."
    )
else:
    partes.append("**Leitura:** o tuning trouxe ganho relevante também no teste, confirmando que vale a pena sintonizar o modelo.")
partes.append("\n**Próximo encontro (14/10):** comparar o Grid Search com o Random Search em tempo de execução e erro.")
display(Markdown("\n".join(partes)))

### Resumo dos encontros 30/09 e 07/10

**Trocas de Óleo**
- 30/09: o Random Forest sem ajustes já reduziu o RMSE do baseline de **8.92** para **3.81**.
- 07/10: o Grid Search escolheu `{'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 50}`. No erro de validação houve **redução de 11.3%**; no teste, o RMSE foi de 3.81 para **3.82** (aumento de 0.5%) e o MAPE teve aumento de 3.9%.

**Manutenção de Motor**
- 30/09: o Random Forest sem ajustes já reduziu o RMSE do baseline de **3.82** para **2.48**.
- 07/10: o Grid Search escolheu `{'max_depth': 10, 'min_samples_leaf': 8, 'n_estimators': 100}`. No erro de validação houve **redução de 8.0%**; no teste, o RMSE foi de 2.48 para **2.39** (redução de 3.5%) e o MAPE teve redução de 2.8%.

**Leitura honesta:** o tuning melhora o erro de validação (que é o que a busca otimiza), mas no teste o ganho adicional é pequeno, sinal de que o Random Forest com hiperparâmetros default já estava perto do teto para estes atributos. O maior salto de desempenho veio de trocar o baseline por um modelo de Machine Learning.

**Próximo encontro (14/10):** comparar o Grid Search com o Random Search em tempo de execução e erro.